## KMS key policies, grants & rotation

KMS access control is **different** from most AWS services. For S3 or DynamoDB, IAM policies alone grant access. For KMS, the **key policy is the primary mechanism** — IAM policies only work if the key policy explicitly delegates to IAM.

### Key policy

A minimal key policy has two pieces:

- A statement **enabling IAM** — `Principal: arn:aws:iam::ACCOUNT:root`, `Action: kms:*` — telling KMS to honour IAM policies for principals in your account. Without it, even an admin's `kms:*` IAM policy does nothing on this key.
- Statements granting **specific principals specific actions** (`kms:Decrypt`, `kms:Encrypt`, `kms:GenerateDataKey`), usually on `Resource: *` since the policy is attached to the key itself.

Cross-account access needs **both sides**: the key policy allows the other account, and that account's IAM delegates to its users.

### Grants

**Grants** delegate specific permissions on a key programmatically, without editing the policy. AWS services use them internally — encrypt an EBS volume and EBS creates a grant permitting decrypt for that one volume. Grants are ideal for temporary, scoped, revocable access.

### Rotation & deletion

- **Automatic rotation** generates new key material **once a year**; the key ID, ARN, and aliases stay the same, so applications see no change. KMS keeps old material, so old data still decrypts. **On-demand rotation** triggers one immediately (use on suspected compromise).
- **Deletion** is never immediate — scheduling it starts a mandatory **7–30 day** window where the key is disabled but recoverable. After it closes, the key is gone and **any data encrypted with it is permanently unrecoverable**. When unsure, **disable** instead — reversible, no data at risk.
- **Multi-Region keys** replicate a key (same key material, same key ID) into other regions, so data encrypted in one region decrypts in another — needed for cross-region DR of encrypted data.